# Train Models

En este notebook, se realizarán los entrenamientos de los modelos candidatos, sin embargo, para ello se debe de tener en cuenta el analisis realziado en `00_analisis_inicial`, el cual resume lo siguiente:

## Función a optmizar

- **TP**: Se evita el 100% de pérdida. **Costo = 0**
- **FP**: Costo de oportunidad = **–25% del monto**
- **FN**: Pérdida = **–100% del monto**
- **TN**: Beneficio = **+25% del monto**
- **J**: Función a optimizar que define el monto

De lo anterior podemos definir la función **J**, la cual será la métrica a optimizar durante el proceso de modelado

$$\boxed{J_{\text{optima}} = -1.00 \cdot FN - 0.25 \cdot FP}$$

> Esta es la función de optimización del modelo. Mide únicamente el **daño financiero causado por los errores del modelo**, ignorando la constante $C$ que el modelo no puede controlar. Su valor es siempre $\leq 0$: cuanto más cercano a cero, mejor el modelo. Si el modelo fuera perfecto ($FN = 0$, $FP = 0$), entonces $J = 0$.

# Modelación para detección de Fraude

En esta sección se realizará el modelado de los datos para la detección de fraude, la cual consta de los siguientes pasos:

- **Train y Test:** en pasos anteriroes especificamente en `02_EDA_train` se realizó la separación de la información para train y test basado en split temporal.
- **Preprocessing pipeline:** en este paso se aplicarán las trasnformaciones y preprocesamientos definidos en el feature engineer para construir las fetaures candidatas y que s epeudan replicar a nivel productivo
- **Train models:** en este paso se realiza el entrenamiento de los modelos candidatos

## Imports y configuración

In [1]:
import sys
import warnings
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

ROOT_DIR = Path.cwd().parent if (Path.cwd().name == "03_model_selection") else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from src.transformers_utils.preprocessing_pipeline import (
    build_preprocessor_from_config,
    load_config, 
    filter_config_by_features,
    build_preprocessor
)
from src.models.model_evaluation import (
    evaluate_fitted_models,
    save_model_artifact,
    extract_fraud_probabilities,
    summarize_pr_curve,
    sweep_business_profit,
    summarize_confusion_matrix,
    sweep_fbeta_score,
    build_threshold_report,
    show_threshold_report,
    plot_precision_recall,
    plot_business_profit,
    plot_confusion_heatmap,
    plot_fbeta_curve,
)


pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATASET_DIR = ROOT_DIR / "dataset"
TRAIN_PATH = DATASET_DIR / "train.csv"
TEST_PATH = DATASET_DIR / "test.csv"
CONFIG_PATH = ROOT_DIR / "src" / "configs" / "preprocessing_config.json"
ARTIFACTS_DIR = ROOT_DIR / "03_model_selection" / "artifacts"

TARGET_COL = "fraude"
# "k" es el identificador 100% único de cada transacción (ver 01_EDA): se excluye
# como feature, no se le aplica ninguna transformación ni se deja como passthrough.
ID_COL = "k"

## Carga de datos y partición train/test

A continuación se cargan los datos de train y test

In [2]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df["fecha"] = pd.to_datetime(train_df["fecha"])
test_df["fecha"] = pd.to_datetime(test_df["fecha"])

X_train = train_df.drop(columns=[TARGET_COL, ID_COL])
y_train = train_df[TARGET_COL]

X_test = test_df.drop(columns=[TARGET_COL, ID_COL])
y_test = test_df[TARGET_COL]

print(f"X_train: {X_train.shape} | y_train: {y_train.shape} | tasa fraude: {y_train.mean():.4f}")
print(f"X_test:  {X_test.shape} | y_test:  {y_test.shape} | tasa fraude: {y_test.mean():.4f}")
X_train.head()

X_train: (104981, 17) | y_train: (104981,) | tasa fraude: 0.0518
X_test:  (45019, 17) | y_test:  (45019,) | tasa fraude: 0.0457


,a,b,c,d,e,f,g,h,j,l,m,n,o,p,fecha,monto,score
0,4,0.7388,6314.50,14.0,0.139279,24.0,BR,7,cat_381751d,2361.0,442.0,1,NaN,Y,2020-03-08 00:02:15,22.18,25
1,4,0.7548,21171.09,20.0,0.514815,7.0,BR,2,cat_a024847,2324.0,73.0,1,NaN,N,2020-03-08 00:04:25,6.00,7
2,4,0.9026,4012.83,50.0,0.274167,1.0,BR,3,cat_1d61c62,235.0,232.0,1,N,Y,2020-03-08 00:08:23,26.67,91
3,4,0.8285,99612.95,1.0,0.000000,4.0,BR,28,cat_01a1725,658.0,0.0,1,N,N,2020-03-08 00:08:39,40.01,91
4,2,0.5992,53526.36,2.0,0.000000,264.0,AR,34,cat_f1e7464,2400.0,10.0,1,NaN,N,2020-03-08 00:09:01,6.25,93


## Pipeline de preprocesamiento de datos

A continuación, se construye el pipeline de procesamiento de datos, los caules quedaron definidos mediante clases construidas con scikit-learn, cabe resaltar los siguientes aspectos importantes:

- La utilización de este tipo de pipelines definidos con scikit-lear permiten utilizarce para datos nuevos, es decir, estos datos podran trasnformarse segun lo aprendido en el proceso de modelado.
- Estos pipelines permiten realizar un desarollo robusto y que generalmente estan enfocados en evitar el data leakage.


### Features a construir por transformer

| Columna origen | Feature de salida | Transformer | Params | Manejo de nulos |
|---|---|---|---|---|
| **a** (0% nulos) | `a_freq_encoded` | `FrequencyEncodingTransformer` | — | No aplica (sin nulos) |
| | `a_rare_grouped` | `RareGroupingTransformer` | — | No aplica (sin nulos) |
| **b** (12.0% nulos) | `b_is_missing` | `MissingFlagTransformer` | — | Señaliza el nulo (flag=1); no lo imputa |
| | `b_is_zero` | `ZeroFlagTransformer` | — | `NaN == 0` → `False` → cae a 0 (efecto colateral, no es un tratamiento real) |
| | `log_b` | `LogFeatureTransformer` | — | Se propaga como NaN |
| | `b_top1_flag` | `TopPercentileFlagTransformer` | `quantile=0.99` | `NaN >= umbral` → `False` → cae a 0 (efecto colateral, no tratamiento real) |
| | `b_qbin_dropna` | `QuantileBinsDropnaTransformer` | `num_bins=5` | Bin dedicado `"__MISSING__"` (tratamiento explícito) |
| **c** (12.0% nulos) | `c_is_missing` | `MissingFlagTransformer` | — | Señaliza el nulo; no lo imputa |
| | `log_c` | `LogFeatureTransformer` | — | Se propaga como NaN |
| | `c_top1_flag` | `TopPercentileFlagTransformer` | `quantile=0.99` | Cae a 0 (mismo efecto colateral que `b_top1_flag`) |
| | `c_qbin_dropna` | `QuantileBinsDropnaTransformer` | `num_bins=5` | Bin dedicado `"__MISSING__"` |
| **d** (0.29% nulos) | `d_is_missing` | `MissingFlagTransformer` | — | Señaliza el nulo; no lo imputa |
| | `d_is_zero` | `ZeroFlagTransformer` | — | Cae a 0 (efecto colateral) |
| | `log_d` | `LogFeatureTransformer` | — | Se propaga como NaN |
| | `d_top1_flag` | `TopPercentileFlagTransformer` | `quantile=0.99` | Cae a 0 (efecto colateral) |
| | `d_qbin_dropna` | `QuantileBinsDropnaTransformer` | `num_bins=5` | Bin dedicado `"__MISSING__"` |
| **e** (0% nulos) | `e_is_zero` | `ZeroFlagTransformer` | — | No aplica (sin nulos) |
| | `log_e` | `LogFeatureTransformer` | — | No aplica (sin nulos) |
| | `e_top1_flag` | `TopPercentileFlagTransformer` | `quantile=0.99` | No aplica (sin nulos) |
| | `e_qbin_dropna` | `QuantileBinsDropnaTransformer` | `num_bins=5` | No aplica (sin nulos) |
| **f** (0.01% nulos) | `f_is_missing` | `MissingFlagTransformer` | — | Señaliza el nulo; no lo imputa |
| | `f_is_zero` | `ZeroFlagTransformer` | — | Cae a 0 (efecto colateral) |
| | `log_f` | `LogFeatureTransformer` | — | Se propaga como NaN |
| | `f_top1_flag` | `TopPercentileFlagTransformer` | `quantile=0.99` | Cae a 0 (efecto colateral) |
| | `f_qbin_dropna` | `QuantileBinsDropnaTransformer` | `num_bins=5` | Bin dedicado `"__MISSING__"` |
| **g** (0.1% nulos) | `g_is_missing` | `MissingFlagTransformer` | — | Señaliza el nulo; no lo imputa |
| | `g_freq_encoded` | `FrequencyEncodingTransformer` | — | `value_counts(dropna=False)`: el nulo es su propia categoría, con su propia frecuencia |
| | `g_rare_grouped` | `RareGroupingTransformer` | — | El nulo es su propia categoría; se agrupa como "rara" solo si su frecuencia es baja |
| **h** (0% nulos) | `h_is_zero` | `ZeroFlagTransformer` | — | No aplica (sin nulos) |
| | `log_h` | `LogFeatureTransformer` | — | No aplica (sin nulos) |
| | `h_top1_flag` | `TopPercentileFlagTransformer` | `quantile=0.99` | No aplica (sin nulos) |
| | `h_qbin_dropna` | `QuantileBinsDropnaTransformer` | `num_bins=5` | No aplica (sin nulos) |
| **j** (0% nulos) | `j_freq_encoded` | `FrequencyEncodingTransformer` | — | No aplica (sin nulos) |
| | `j_rare_grouped` | `RareGroupingTransformer` | — | No aplica (sin nulos) |
| | `j_target_enc` | `TargetEncodingCVTransformer` | — | No aplica (sin nulos) |
| **l** (0.01% nulos) | `l_is_missing` | `MissingFlagTransformer` | — | Señaliza el nulo; no lo imputa |
| | `l_is_zero` | `ZeroFlagTransformer` | — | Cae a 0 (efecto colateral) |
| | `log_l` | `LogFeatureTransformer` | — | Se propaga como NaN |
| | `l_top1_flag` | `TopPercentileFlagTransformer` | `quantile=0.99` | Cae a 0 (efecto colateral) |
| | `l_qbin_dropna` | `QuantileBinsDropnaTransformer` | `num_bins=5` | Bin dedicado `"__MISSING__"` |
| **m** (0.29% nulos) | `m_is_missing` | `MissingFlagTransformer` | — | Señaliza el nulo; no lo imputa |
| | `m_is_zero` | `ZeroFlagTransformer` | — | Cae a 0 (efecto colateral) |
| | `log_m` | `LogFeatureTransformer` | — | Se propaga como NaN |
| | `m_top1_flag` | `TopPercentileFlagTransformer` | `quantile=0.99` | Cae a 0 (efecto colateral) |
| | `m_qbin_dropna` | `QuantileBinsDropnaTransformer` | `num_bins=5` | Bin dedicado `"__MISSING__"` |
| **o** (72.44% nulos) | `o_freq_encoded` | `FrequencyEncodingTransformer` | — | El nulo es su propia categoría (72.44% de frecuencia) |
| | `o_target_enc` | `TargetEncodingCVTransformer` | — | El nulo es su propia categoría con su propia tasa de fraude aprendida (`groupby(..., dropna=False)`) — **no** cae al promedio global |
| **fecha** (0% nulos) | `hour_fecha` | `HourTransformer` | — | No aplica (sin nulos); si hubiera NaT, quedaría NaN |
| | `weekday_fecha` | `WeekdayTransformer` | — | No aplica (sin nulos); si hubiera NaT, quedaría NaN |
| | `is_weekend_fecha` | `IsWeekendFlagTransformer` | — | No aplica (sin nulos); si hubiera NaT, caería a 0 (no distinguiría "sin fecha" de "no es fin de semana") |
| | `hour_sin_fecha` | `HourCyclicSinCosTransformer` | `component="sin"` | No aplica (sin nulos); si hubiera NaT, quedaría NaN |
| | `hour_cos_fecha` | `HourCyclicSinCosTransformer` | `component="cos"` | No aplica (sin nulos); si hubiera NaT, quedaría NaN |
| **monto** (0% nulos) | `log_monto` | `LogFeatureTransformer` | — | No aplica (sin nulos) |
| | `monto_top1_flag` | `TopPercentileFlagTransformer` | `quantile=0.99` | No aplica (sin nulos) |
| | `monto_qbin_dropna` | `QuantileBinsDropnaTransformer` | `num_bins=5` | No aplica (sin nulos) |
| **score** (0% nulos) | `score_is_zero` | `ZeroFlagTransformer` | — | No aplica (sin nulos) |
| | `log_score` | `LogFeatureTransformer` | — | No aplica (sin nulos) |
| | `score_top1_flag` | `TopPercentileFlagTransformer` | `quantile=0.99` | No aplica (sin nulos) |
| | `score_qbin_dropna` | `QuantileBinsDropnaTransformer` | `num_bins=5` | No aplica (sin nulos) |
| **d + m** | `pca_dm_1` | `PCATransformer` | `num_components=1`, `prefix="pca_dm"` | **Imputación real**: mediana de train por columna (`fill_strategy="median"`), aprendida en `fit` |
| **monto + l** | `monto_per_l` | `RatioTransformer` | — | Se propaga NaN si `monto` o `l` es nulo; si `l=0` (no nulo) usa `default_value` (NaN por defecto) |
| **n, p** (0% nulos) | `n`, `p` | *(sin transformer — passthrough)* | `remainder="passthrough"` | No aplica (columnas sin nulos, pasan sin transformar) |
| **k** | *(excluida)* | *(ninguno — se descarta como ID antes de entrar al pipeline)* | — | No aplica (columna excluida del pipeline) |

Total: 61 features nuevas + 2 passthrough (`n`, `p`) = 63 columnas de salida.

### Nulos en las variables originales (resumen)

| Variable original | % nulos (train) | ¿Cómo se maneja ese nulo en el pipeline? |
|---|---|---|
| `a`, `e`, `h`, `j`, `n`, `p`, `fecha`, `monto`, `score` | 0% | No aplica — no hay nulos que tratar |
| `b`, `c` | 12.0% | Se señaliza con flag de nulo y se les da un bin `"__MISSING__"` dedicado; las derivadas numéricas (`log_*`) propagan NaN, no se imputan |
| `d`, `f`, `l`, `m` | 0.01%–0.29% | Igual que `b`/`c`: flag + bin dedicado; sin imputación numérica real |
| `g` | 0.1% | Se señaliza con flag; además se trata como categoría propia en el encoding de frecuencia y en el agrupamiento de categorías raras |
| `o` | 72.44% | **No se imputa deliberadamente**: se trata como categoría propia con su propia frecuencia y su propia tasa de fraude aprendida — es la señal más fuerte del dataset y perderla equivaldría a destruir la variable |
| `k` | — | Columna excluida del pipeline (identificador, no es feature) |

**Conclusión**: ningún transformer imputa un valor numérico "de relleno" salvo `PCATransformer` (mediana). El resto de la estrategia es deliberada: preservar la señal de "faltante" como flag o como categoría propia, en lugar de rellenarla — coherente con la conclusión del EDA de que el nulo de `o` es la variable más discriminante del dataset.

In [3]:
preprocessor = build_preprocessor_from_config(CONFIG_PATH)

X_train_transformed = preprocessor.fit_transform(X_train, y_train)
X_test_transformed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_prep = pd.DataFrame(X_train_transformed, columns=feature_names, index=X_train.index)
X_test_prep = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test.index)

print(f"X_train_prep: {X_train_prep.shape}")
print(f"y_train: {y_train.shape}")
print("")
print(f"X_test_prep:  {X_test_prep.shape}")
print(f"y_test:  {y_test.shape}")

X_train_prep.head()

X_train_prep: (104981, 73)
y_train: (104981,)

X_test_prep:  (45019, 73)
y_test:  (45019,)


,a_freq_encoded,a_rare_grouped,b_is_missing,b_is_zero,log_b,b_top1_flag,b_qbin_dropna,c_is_missing,log_c,c_top1_flag,c_qbin_dropna,d_is_missing,d_is_zero,log_d,d_top1_flag,d_qbin_dropna,e_is_zero,log_e,e_top1_flag,e_qbin_dropna,f_is_missing,f_is_zero,log_f,f_top1_flag,f_qbin_dropna,g_is_missing,g_freq_encoded,g_rare_grouped,h_is_zero,log_h,h_top1_flag,h_qbin_dropna,j_freq_encoded,j_rare_grouped,j_target_enc,l_is_missing,l_is_zero,log_l,l_top1_flag,l_qbin_dropna,m_is_missing,m_is_zero,log_m,m_top1_flag,m_qbin_dropna,o_freq_encoded,o_target_enc,hour_fecha,weekday_fecha,is_weekend_fecha,hour_sin_fecha,hour_cos_fecha,log_monto,monto_top1_flag,monto_qbin_dropna,score_is_zero,log_score,score_top1_flag,score_qbin_dropna,pca_dm_1,monto_per_l,monto_sum_10d,monto_mean_10d,monto_median_10d,monto_max_10d,monto_min_10d,monto_sum_15d,monto_mean_15d,monto_median_15d,monto_max_15d,monto_min_15d,n,p
0,0.850821,4,0,0,0.553195,0,2,0,8.750762,0,1,0,0,2.70805,0,2,0,0.130395,0,0,0,0,3.218876,0,3,0,0.755489,BR,0,2.079442,0,1,0.004563,__OTHER__,0.042624,0,0,7.767264,0,2,0,0,6.09357,0,3,0.724388,0.021018,0,6,1,0.0,1.0,3.14329,0,2,0,3.258097,0,1,142.485292,0.009394,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Y
1,0.850821,4,0,0,0.562355,0,2,0,9.960439,0,1,0,0,3.044522,0,2,0,0.415293,0,2,0,0,2.079442,0,2,0,0.755489,BR,0,1.098612,0,0,0.000133,__OTHER__,0.023563,0,0,7.751475,0,2,0,0,4.304065,0,1,0.724388,0.021533,0,6,1,0.0,1.0,1.94591,0,0,0,2.079442,0,0,-226.0406,0.002582,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,N
2,0.850821,4,0,0,0.643221,0,4,0,8.297501,0,0,0,0,3.931826,1,3,0,0.242292,0,1,0,0,0.693147,0,0,0,0.755489,BR,0,1.386294,0,1,0.002648,__OTHER__,0.044569,0,0,5.463832,0,0,0,0,5.451038,0,2,0.113421,0.230557,0,6,1,0.0,1.0,3.320349,0,2,0,4.521789,0,4,-66.040159,0.113489,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Y
3,0.850821,4,0,0,0.603496,0,4,0,11.509057,0,3,0,0,0.693147,0,0,1,0.0,0,0,0,0,1.609438,0,1,0,0.755489,BR,0,3.367296,0,4,0.000438,__OTHER__,0.119965,0,0,6.490724,0,0,0,1,0.0,0,0,0.113421,0.228374,0,6,1,0.0,1.0,3.713816,0,3,0,4.521789,0,4,-299.693151,0.060805,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,N
4,0.101837,2,0,0,0.469504,0,0,0,10.887948,0,2,0,0,1.098612,0,0,1,0.0,0,0,0,0,5.57973,0,4,0,0.204132,AR,0,3.555348,0,4,0.000076,__OTHER__,0.032399,0,0,7.783641,0,2,0,0,2.397895,0,0,0.724388,0.021018,0,6,1,0.0,1.0,1.981001,0,0,0,4.543295,0,4,-289.663017,0.002604,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,N


## Modelos candidatos

Para definir un modelo final, es necesario abordar la pregunta sobre que tipo de modelo es el adecuado. En la literatura de detección de fraude los modelos basados en árboles como RandomForest XGBoost, LightGBM y CatBoost son en egenral el caballo de batalla mas común en datos tabulares para clasificación de fraude, estos modelos capturan interacciones no lineales y toleran el desbalance.

Se puede destacar lo siguiente de los modelos mencionados:\

- **XGBoost** es el más robusto y battle-tested — el estándar de la industria, documentación enorme y muy estable. Por su ronustez, este modelo es mas lento, pero ofrece mayor confiabilidad y soporte por encima de la velocidad.

- **CatBoost** es el más fácil de usar bien de entrada, sus valores por defecto suelen dar buenos resultados sin tuning, y es el mejor manejando variables categóricas de alta cardinalidad. Su desventaja radica en que su entrenamiento es más lento y menos flexible en tuning.

- **LightGBM** es el que mejor reparte el balance entre velocidad, rendimiento predictivo, uso de memoria, escalabilidad y facilidad de uso, sin embargo este modelo al ser mas simple que un XGBoost, puede ser propenso a overfitting y a sensibilidad con fetaures categoricas.

- **Random Forest** es el más simple y difícil de sobreajustar, buen baseline no lineal, pero en general el menos equilibrado del grupo, suele quedar por debajo de los tres boosting en precisión y no escala tan bien. Sigue siendo útil como sanity-check.

Refrencias bibliograficas de apoyo:

- Grinsztajn et al., Why do tree-based models still outperform deep learning on tabular data? (NeurIPS 2022) — evidencia de por qué los GBDT dominan en datos tabulares. https://arxiv.org/abs/2207.08815

- Impact of Sampling Techniques and Data Leakage on XGBoost Performance in Credit Card Fraud Detection (2024)- evidencia que los mdoelso basados en arboles como XGBoost pueden ser propensos a overfitting si hay Data Leakage y si se realiza un mal data sampling https://arxiv.org/pdf/2412.07437


**Notas adicionales**

De los modelos anteriores se peuden destacar riesgos que peuden verse reflejados a nivel productivo:

1. Concept drift y degradación silenciosa. Es el riesgo número uno en fraude: los defraudadores cambian de táctica y la distribución de datos se corre con el tiempo. El modelo sigue prediciendo con normalidad pero su desempeño real cae, y como los árboles no extrapolan bien fuera del rango visto en entrenamiento, ante patrones nuevos fallan sin avisar. Exige monitorear drift (de features y de la tasa de fraude) y reentrenar periódicamente.

2. Overfitting y data leakage que solo se ven en producción. Un modelo que en test lucía excelente puede desplomarse en real por dos causas: árboles demasiado profundos/numerosos que memorizaron, o leakage (features que en entrenamiento contenían información del futuro o del resultado). LightGBM es especialmente propenso al overfitting por su crecimiento leaf-wise. El síntoma clásico: métricas de validación muy altas que no se reproducen en vivo.

3. Latencia, tamaño y costo en inferencia. Un ensemble con cientos o miles de árboles puede ser pesado para scoring en tiempo real (crítico si se evaluan transacciones online). Random Forest tiende a ser el más lento y voluminoso; XGBoost/LightGBM/CatBoost son más ágiles pero igual crecen con n_estimators. Se recomiendda vigilar latencia p99, memoria y versionado del artefacto.


## Model Baseline

Con Base a lo anterior, si bien los mdoelso basados en arboles son buenso candidatos para la predicción de fraude, es necesario definir un modelo baseline inicial, en general el mas recomendado como punto de partida es la Regresión logística, se considera como estandar comparativo para contrastar resultados VS modelos basados en arbolesde lo cual se puede destacar:

- Linealidad y simplicidad: al ser menos complejo establece la base de comparación, no siempre un mdoelo mas complejo o robusto termian siendo la mejro opción.
- Interpretabilidad: los coeficientes dicen qué features pesan mas que otras, buen sanity-check antes de confiar en un árbol.
- Entrenamiento y velocidad: modelo simple que facilita su implementación rápida.


Para entrenar el model BAseline y los modelos absados en árboles mencionado anteriormente, se ajsutan dichos mdoelos con los siguientes grupos de fetaures:

- Features sopriginales, descartando la columna tipo ID y la columan de fecha:
    - `a`, `b`, `c`, `d`, `e`, `f`, `g`, `h`,`j`, `l`, `m`, `n`, `o`, `p`, `monto`, `score`
- Features elegidas según el análisis realizado en 01_statistical_analysis:
    - `g_freq_encoded`, `j_freq_encoded`, `j_target_enc`, `o_freq_encoded`, `o_target_enc`, `score_qbin_dropna`, `f_is_zero`, `pca_dm_1`, `monto_per_l`
- Features de agregación:
    - `monto_sum_10d`, `monto_mean_10d`, `monto_median_10d`, `monto_max_10d`, `monto_min_10d`, `monto_min_10d`, `monto_sum_15d`, `monto_mean_15d`, `monto_median_15d`, `monto_max_15d`, `monto_min_15d`



### Selección final de features

Se define un único conjunto de 32 features, **igual para los 5 modelos**, aplicando el siguiente criterio sobre encoding vs. variable cruda:

- **Categóricas de alta/moderada cardinalidad** (`g`, `j`, `o`): se reemplaza la variable cruda por su encoding — un one-hot de `j` (7,558+ categorías) es inviable, y mantener ambas (cruda + encoded) para `g`/`o` sería redundante.
- **Binarias** (`n`, `p`): solo se usa el valor crudo. El frequency/target encoding de una binaria es una función biyectiva del valor crudo (no agrega información nueva) — ya documentado como decisión deliberada en `feature_engineer_config.yml`.
- **Baja cardinalidad y señal débil** (`a`): se mantiene cruda (con one-hot barato de 4 categorías más adelante), no vale la pena la encoded dado que su señal es marginal.
- **El resto de features numéricas seleccionadas** (`f_is_zero`, `score_qbin_dropna`, `pca_dm_1`, `monto_per_l`) no son encodings de categóricas — son flags/bins/interacciones que sí aportan información distinta a la variable cruda, así que se mantienen **junto con** su columna original (`f`, `score`, `d`+`m`, `monto`+`l`).

In [4]:

RAW_PASSTHROUGH = ["a", "b", "c", "d", "e", "f", "h", "l", "m", "n", "p", "monto", "score"]
ENCODED_CATEGORICAL = ["g_freq_encoded", "j_freq_encoded", "j_target_enc", "o_freq_encoded", "o_target_enc"]
SELECTED_ENGINEERED = ["score_qbin_dropna", "f_is_zero", "pca_dm_1", "monto_per_l"]
WINDOW_AGG = [
    "monto_sum_10d", "monto_mean_10d", "monto_median_10d", "monto_max_10d", "monto_min_10d",
   "monto_sum_15d", "monto_mean_15d", "monto_median_15d", "monto_max_15d", "monto_min_15d",
]

WINDOW_AGG = [
    "monto_sum_15d"
]

TARGET_FEATURES = ENCODED_CATEGORICAL + SELECTED_ENGINEERED + WINDOW_AGG

general_config = load_config(CONFIG_PATH)
baseline_config = filter_config_by_features(general_config, TARGET_FEATURES, passthrough_cols=RAW_PASSTHROUGH)
baseline_preprocessor = build_preprocessor(baseline_config)

baseline_preprocessor.fit(X_train, y_train)
baseline_feature_names = list(baseline_preprocessor.get_feature_names_out())

print(f"Steps mantenidos del config general: {len(baseline_config['steps'])}")
print(f"Total de features del baseline: {len(baseline_feature_names)}")
print(sorted(baseline_feature_names))

Steps mantenidos del config general: 16
Total de features del baseline: 23
['a', 'b', 'c', 'd', 'e', 'f', 'f_is_zero', 'g_freq_encoded', 'h', 'j_freq_encoded', 'j_target_enc', 'l', 'm', 'monto', 'monto_per_l', 'monto_sum_15d', 'n', 'o_freq_encoded', 'o_target_enc', 'p', 'pca_dm_1', 'score', 'score_qbin_dropna']


In [5]:
CATEGORICAL_COLS = ["a", "p", "score_qbin_dropna"]
cat_idx = [baseline_feature_names.index(c) for c in CATEGORICAL_COLS]
num_idx = [i for i in range(len(baseline_feature_names)) if i not in cat_idx]

post_encoder = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_idx),
    ("num", SimpleImputer(strategy="median"), num_idx),
])

### Entrenamiento de los 5 modelos candidatos

En este apartado se definen los pieplines de cada modelo y sus parametros base, ademas en el path `03_model_selection/artifacts`, se guardan los artefactos de cada modelo, esto en caso de que se requiera implementar en producción y sea replicable enb otros ambientes o disponibilizarlo como API.

In [6]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
DECISION_THRESHOLD = 0.5

In [7]:

models = {
    "LogisticRegression": LogisticRegression(
        max_iter = 1000, 
        class_weight = "balanced"
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators = 100, 
        class_weight = "balanced", 
        random_state = 42, 
        n_jobs = -1
    ),
    "LightGBM": LGBMClassifier(
        random_state = 42, 
        objective = "binary",
        max_depth = 10,
        scale_pos_weight = scale_pos_weight-10,
        verbosity = -1
    ),
    "CatBoost": CatBoostClassifier(
        random_state = 42, 
        verbose = 0, 
        scale_pos_weight = scale_pos_weight-10,
        eval_metric="Precision",
        depth=5
    ),
    "XGBoost": XGBClassifier(
        random_state = 42,
        objective = 'binary:logistic', 
        scale_pos_weight = scale_pos_weight-8,
        booster = "gbtree",
        max_depth = 5,
        subsample = 0.8,         
        colsample_bytree = 0.8, 
        n_estimators = 2000,
        eval_metric = "pre",
        n_jobs = -1,
        learning_rate = 0.05
    ),
}

fitted_pipelines = {}
monto_test = test_df["monto"].to_numpy()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for name, clf in models.items():
        pipe = Pipeline([
            ("preprocessor", clone(baseline_preprocessor)),
            ("post_encode", clone(post_encoder)),
            ("clf", clf),
        ])
        pipe.fit(X_train, y_train)
        fitted_pipelines[name] = pipe

        artifact_paths = save_model_artifact(pipe, name, ARTIFACTS_DIR)
        print(f"{name}: {artifact_paths}")

results_df = evaluate_fitted_models(fitted_pipelines, X_test, y_test, monto=monto_test, threshold=DECISION_THRESHOLD)
results_df


LogisticRegression: {'pipeline': '/Users/dpiedrahita/proyectos/DS_pro/03_model_selection/artifacts/LogisticRegression/pipeline.joblib', 'metadata': '/Users/dpiedrahita/proyectos/DS_pro/03_model_selection/artifacts/LogisticRegression/metadata.json'}
RandomForest: {'pipeline': '/Users/dpiedrahita/proyectos/DS_pro/03_model_selection/artifacts/RandomForest/pipeline.joblib', 'metadata': '/Users/dpiedrahita/proyectos/DS_pro/03_model_selection/artifacts/RandomForest/metadata.json'}
LightGBM: {'pipeline': '/Users/dpiedrahita/proyectos/DS_pro/03_model_selection/artifacts/LightGBM/pipeline.joblib', 'native': '/Users/dpiedrahita/proyectos/DS_pro/03_model_selection/artifacts/LightGBM/model.txt', 'metadata': '/Users/dpiedrahita/proyectos/DS_pro/03_model_selection/artifacts/LightGBM/metadata.json'}
CatBoost: {'pipeline': '/Users/dpiedrahita/proyectos/DS_pro/03_model_selection/artifacts/CatBoost/pipeline.joblib', 'native': '/Users/dpiedrahita/proyectos/DS_pro/03_model_selection/artifacts/CatBoost/mod

/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,AUC,PR-AUC,Precision (TH fijo),Recall (TH fijo),J ($),J ($) block all,TH óptimo,Precision (TH óptimo),Recall (TH óptimo),J óptimo ($)
modelo,,,,,,,,,,
LogisticRegression,0.7525,0.1907,0.1065,0.6691,-210733.2775,-428970.0425,0.7525,0.3143,0.1788,-127170.6825
RandomForest,0.8673,0.4049,0.5374,0.3421,-105370.8000,-428970.0425,0.3010,0.3452,0.4985,-93752.2425
LightGBM,0.8850,0.4354,0.2806,0.6156,-91519.2750,-428970.0425,0.6120,0.3702,0.5005,-86068.1375
CatBoost,0.8805,0.4368,0.3045,0.5758,-91003.7225,-428970.0425,0.5418,0.3383,0.5418,-87886.9525
XGBoost,0.8681,0.4203,0.3548,0.4879,-91423.7725,-428970.0425,0.5619,0.3956,0.4495,-90194.0975
